In [1]:
ConnectString = "mysql -h database-1.clqxqhhe6wft.us-east-1.rds.amazonaws.com -P 3306 -u admin -p'<Enter_DB_Password>' --ssl-verify-server-cert  --ssl-ca=/certs/global-bundle.pem mysql"

host=ConnectString[9:60]
user='admin'
password='Data608-Project'
database='' # Optional: leave out to create DB first


In [2]:
#!pip install mysql-connector-python --break-system-packages

In [3]:
import mysql.connector
from mysql.connector import Error

In [4]:
connection = mysql.connector.connect(
    host=host,
    user=user,
    password=password,
    database='' # Optional: leave out to create DB first
   )

if connection.is_connected():
    db_info = info = connection.server_info
    print(f"Connected to MySQL Server version: {db_info}")
    cursor = connection.cursor()


Connected to MySQL Server version: 8.4.7


In [5]:

# 2. Create a cursor
cursor = connection.cursor()

# 3. Execute the CREATE DATABASE command
# Use "IF NOT EXISTS" to prevent errors if the DB already exists
cursor.execute("CREATE DATABASE IF NOT EXISTS CHESSBOT")

print("Database created successfully!")

# 4. (Optional) Switch to the new database to start using it
#cursor.execute("USE my_new_database")
# Alternatively, re-assign the database property directly:
# mydb.database = 'my_new_database'


Database created successfully!


In [6]:
# 1. Execute the command
cursor.execute("SHOW DATABASES")

# 2. Fetch and print the results
print("Available Databases:")
for (db_name,) in cursor:
    print(f"- {db_name}")

Available Databases:
- CHESSBOT
- information_schema
- mysql
- performance_schema
- sys


In [7]:
#select database to use
cursor = connection.cursor()
cursor.execute("USE CHESSBOT")
print("Selected CHESSBOT database")

Selected CHESSBOT database


In [8]:
#cursor.execute("DROP TABLE IF EXISTS files")
#cursor.execute("DROP TABLE IF EXISTS session")
#cursor.execute("DROP TABLE IF EXISTS moves")
#cursor.execute("DROP TABLE IF EXISTS games")

In [9]:
# --- CREATE: Create a table ---
games_table_name = 'games'
moves_table_name = 'moves'
files_table_name = 'files'

gamestableexists = False
movestableexists = False
filestableexists = False

# 1. Execute the command to check if table exists
cursor.execute("SHOW TABLES")

# 2. Fetch and print the results, capture table exists
print("Available Tables:")
for (db_name,) in cursor:
    print(f"- {db_name}")
    if(db_name == games_table_name):
        gamestableexists = True
        print ("-------------")
        print ("table: ",games_table_name," already exists so will not recreate")
    if(db_name == moves_table_name):
        movestableexists = True
        print ("-------------")
        print ("table: ",moves_table_name," already exists so will not recreate")
    if(db_name == files_table_name):
        filestableexists = True
        print ("-------------")
        print ("table: ",files_table_name," already exists so will not recreate")


if gamestableexists == False:   # create games table
    print ("-------------")
    print ("table: ",games_table_name," does not exists")
    CreateTable = "CREATE TABLE " + games_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,Site VARCHAR(50),White_ELO INT,Black_ELO INT,Opening VARCHAR(100))"
    print(CreateTable)
    cursor.execute(CreateTable)
    print("Table ",games_table_name, " created.")    

if movestableexists == False:   # create moves table
    print ("-------------")
    print ("table: ",moves_table_name," does not exists")
    CreateTable2 = "CREATE TABLE " + moves_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,Set5 VARCHAR(75),Set10 VARCHAR(75),Set15 VARCHAR(75),Set20 VARCHAR(75),Set30 VARCHAR(150),Set40 VARCHAR(150),Set70 VARCHAR(450),Set_Remain VARCHAR(450),BlackELO INT,Game_id INT,FOREIGN KEY (Game_id) REFERENCES games(id))"
    print(CreateTable2)
    cursor.execute(CreateTable2)
    print("Table ",moves_table_name, " created.")    

    ### create Move indexes
    cursor.execute("CREATE INDEX idx_set5 ON moves(Set5)")
    cursor.execute("CREATE INDEX idx_set10 ON moves(Set10)")
    cursor.execute("CREATE INDEX idx_set15 ON moves(Set15)")
    cursor.execute("CREATE INDEX idx_set20 ON moves(Set20)")
    cursor.execute("CREATE INDEX idx_set30 ON moves(Set30)")
    print ("-------------")
    print ("Moves table indexes are created")

if filestableexists == False:   # create files table
    print ("-------------")
    print ("table: ",files_table_name," does not exists")
    CreateTable3 = "CREATE TABLE " + files_table_name + "(id INT AUTO_INCREMENT PRIMARY KEY,filename VARCHAR(100),DateTimeLoaded VARCHAR(50),NumGamesTotal INT,NumGamesIngested INT,Month_value VARCHAR(3),Year_value VARCHAR(5),File_size VARCHAR(50))"
    print(CreateTable3)
    cursor.execute(CreateTable3)
    print("Table ",files_table_name, " created.")    


Available Tables:
-------------
table:  games  does not exists
CREATE TABLE games(id INT AUTO_INCREMENT PRIMARY KEY,Site VARCHAR(50),White_ELO INT,Black_ELO INT,Opening VARCHAR(100))
Table  games  created.
-------------
table:  moves  does not exists
CREATE TABLE moves(id INT AUTO_INCREMENT PRIMARY KEY,Set5 VARCHAR(75),Set10 VARCHAR(75),Set15 VARCHAR(75),Set20 VARCHAR(75),Set30 VARCHAR(150),Set40 VARCHAR(150),Set70 VARCHAR(450),Set_Remain VARCHAR(450),BlackELO INT,Game_id INT,FOREIGN KEY (Game_id) REFERENCES games(id))
Table  moves  created.
-------------
Moves table indexes are created
-------------
table:  files  does not exists
CREATE TABLE files(id INT AUTO_INCREMENT PRIMARY KEY,filename VARCHAR(100),DateTimeLoaded VARCHAR(50),NumGamesTotal INT,NumGamesIngested INT,Month_value VARCHAR(3),Year_value VARCHAR(5),File_size VARCHAR(50))
Table  files  created.


In [10]:
#check games table

cursor.execute("DESCRIBE "+games_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
Site                 varchar(50)     YES             
White_ELO            int             YES             
Black_ELO            int             YES             
Opening              varchar(100)    YES             


In [11]:
#check moves table

cursor.execute("DESCRIBE "+moves_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
Set5                 varchar(75)     YES        MUL  
Set10                varchar(75)     YES        MUL  
Set15                varchar(75)     YES        MUL  
Set20                varchar(75)     YES        MUL  
Set30                varchar(150)    YES        MUL  
Set40                varchar(150)    YES             
Set70                varchar(450)    YES             
Set_Remain           varchar(450)    YES             
BlackELO             int             YES             
Game_id              int             YES        MUL  


In [12]:
#check files table

cursor.execute("DESCRIBE "+files_table_name)

# Fetch and print the headers
print(f"{'Field':<20} {'Type':<15} {'Null':<10} {'Key':<5}")
print("-" * 50)

for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]:<5}")

Field                Type            Null       Key  
--------------------------------------------------
id                   int             NO         PRI  
filename             varchar(100)    YES             
DateTimeLoaded       varchar(50)     YES             
NumGamesTotal        int             YES             
NumGamesIngested     int             YES             
Month_value          varchar(3)      YES             
Year_value           varchar(5)      YES             
File_size            varchar(50)     YES             


In [13]:
# --- READ: Fetch the record count of games table ---
cursor.execute(f"SELECT COUNT(*) FROM {games_table_name}")
count = cursor.fetchone()[0]
print(f"{games_table_name} table has entries: {count}")


games table has entries: 0


In [14]:
# --- READ: Fetch the record count of moves table ---
cursor.execute(f"SELECT COUNT(*) FROM {moves_table_name}")
count = cursor.fetchone()[0]
print(f"{moves_table_name} table has entries: {count}")


moves table has entries: 0


In [15]:
# --- READ: Fetch the record count of files table ---
cursor.execute(f"SELECT COUNT(*) FROM {files_table_name}")
count = cursor.fetchone()[0]
print(f"{files_table_name} table has entries: {count}")


files table has entries: 0


In [16]:
#Michael's sandbox..
#INSERT

insert_query = "INSERT INTO " + games_table_name + "(Site, White_ELO,Black_ELO,Opening) VALUES (%s, %s, %s, %s)"
cursor.execute(insert_query, ("https://lichess.org/gx3qb4ur", 1155,1096,"King's Indian Attack"))
connection.commit()  # Required to save changes
InsertedRecord = cursor.lastrowid
print(f"Record inserted. ID: {InsertedRecord}")


Record inserted. ID: 1


In [17]:
#Michael's sandbox..
#READ

sqlstatement = "SELECT * FROM moves WHERE  Set5 = '1. e2e4 e7e5 2. g1f3 b8c6 3. f1c4 g8f6 4. f3g5 d7d5 5. e4d5 f6d5' AND Set10 LIKE '6. g5f7 e8f7 7. d1f3 f7e8 8. c4d5 c6d4 9. f3f7%'"
cursor.execute(sqlstatement)
records = cursor.fetchall()
print(f"Total rows: {cursor.rowcount}")
for row in records:
    print("---------------")
    for value in row:
        print(str(value))



Total rows: 0


In [18]:
#Michael's sandbox..
#READ

sqlstatement = "SELECT * FROM games WHERE  id = '35'"
cursor.execute(sqlstatement)
records = cursor.fetchall()
print(f"Total rows: {cursor.rowcount}")
for row in records:
    print("---------------")
    for value in row:
        print(str(value))



Total rows: 0


In [19]:
#Michael's sandbox..
#UPDATE

update_query = "UPDATE games SET Opening = %s WHERE id = %s"
cursor.execute(update_query, ("Some new death method", InsertedRecord))
connection.commit()
print("Record updated.")


Record updated.


In [20]:
#Michael's sandbox..
#DELETE

delete_query = "DELETE FROM games WHERE id = %s"
cursor.execute(delete_query, (InsertedRecord,))
connection.commit()
print("Record deleted.")


Record deleted.
